In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets
from torchvision.transforms import v2
import torchvision.transforms.functional as FF
import torch.nn.functional as F
import random
import math
from PIL import Image

In [92]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

learning_rate = 3e-3
epochs = 5

Using mps device


In [83]:
training_data = datasets.VOCDetection(
    root="data",
    year="2007",
    image_set="trainval",
    download=True,
    transform=v2.Compose([
        v2.ToImage(), v2.ToDtype(torch.float32, scale=True)
    ])
)

In [84]:
class PatchesDataset(Dataset):
    def __init__(self, image_set = "trainval"):
        self.voc = datasets.VOCDetection(
            root="data", year="2007", image_set=image_set, download=True
        )
        self.PatchSize = 96
        self.gap = 48
        self.jitter = 7
        self.GridSize = 3

        self.GridSpan = (self.PatchSize * self.GridSize) + (self.gap * (self.GridSize - 1))

        self.MinSize = self.GridSpan + (self.jitter*2)

        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]

    def _resizePreserveAspect(self, img):
        w,h = img.size
        TotalPixels = w*h

        TargetPixels = random.uniform(160000, 450000)

        ScaleFactor = math.sqrt(TargetPixels/TotalPixels)
        newW = int(w * ScaleFactor)
        newH = int(h * ScaleFactor)

        if newW < self.MinSize or newH < self.MinSize:
            ScaleUp = self.MinSize / min(newW, newH)
            newW = math.ceil(newW * ScaleUp)
            newH = math.ceil(newH * ScaleUp)

        return img.resize((newW, newH), Image.BILINEAR)
    
    def _dropColor(self, patch):
        if random.random() < 0.5:
            patch = FF.to_grayscale(patch, num_output_channels=3)
        return patch
    
    def _pixelate(self, patch):
        if random.random() < 0.2:
            downSize = random.randint(10, self.PatchSize - 1)
            patch = patch.resize((downSize, downSize), Image.BILINEAR)
            patch = patch.resize((self.PatchSize, self.PatchSize), Image.BILINEAR)
        return patch
    
    def __len__(self):
        return len(self.voc)
    
    def __getitem__(self, idx):
        img, _ = self.voc[idx]
        img = self._resizePreserveAspect(img)
        w, h = img.size

        startX = random.randint(self.jitter, w - self.GridSpan - self.jitter)
        startY = random.randint(self.jitter, h - self.GridSpan - self.jitter)
        patches = []

        for i in range(self.GridSize):
            for j in range(self.GridSize):
                baseX = startX + j * (self.PatchSize + self.gap)
                baseY = startY + i * (self.PatchSize + self.gap)

                dx = random.randint(-self.jitter, self.jitter)
                dy = random.randint(-self.jitter, self.jitter)

                xCrop = baseX + dx
                yCrop = baseY + dy
                patch = img.crop((xCrop, yCrop,
                                  xCrop + self.PatchSize,
                                  yCrop + self.PatchSize))
                
                patch = self._dropColor(patch)
                patch = self._pixelate(patch)

                patchTensor = FF.to_tensor(patch)

                patchTensor = FF.normalize(patchTensor, mean=self.mean, std=self.std)

                patches.append(patchTensor)
        
        validIndices = [0, 1, 2, 3, 5, 6, 7, 8]
        label = random.randint(0, 7)
        contextIdx = validIndices[label]
        

        return patches[4], patches[contextIdx], label


In [85]:
# Test
dataset = PatchesDataset()
    
# Cho vào Dataloader
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Thử lấy 1 batch
for center_patches, context_patches, labels in dataloader:
    print("Center patches shape :", center_patches.shape)
    print("Context patches shape:", context_patches.shape)
    print("Labels shape         :", labels.shape)
    
    # In thử vài nhãn để kiểm tra
    print("First 5 labels       :", labels[:5].tolist())
    break

Center patches shape : torch.Size([32, 3, 96, 96])
Context patches shape: torch.Size([32, 3, 96, 96])
Labels shape         : torch.Size([32])
First 5 labels       : [1, 1, 2, 5, 1]


In [86]:
training_data = PatchesDataset(image_set="trainval")
test_data = PatchesDataset(image_set="test")


training_dataloader = DataLoader(training_data, batch_size=32)
test_dataloader = DataLoader(test_data, batch_size=32)

In [ ]:
class LateFusionNetwork(nn.Module):
    def __init__(self, num_classes: int = 8, dropout_rate: float=0.5):
        # Input : 96x96x3
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 96, 11, 4),     # conv1 99x99x96
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2),             # pool1 49x49x96
            nn.BatchNorm2d(96),             # bn1

            nn.Conv2d(96, 384, 5, 2),    # conv2 23x23x384
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2),             # pool2 11x11x384
            nn.BatchNorm2d(384),            # bn2

            nn.Conv2d(384, 384, 3, 1),   # conv3 9x9x385
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 384, 3, 1),   # conv4 7x7x384
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, 3, 2),   # conv5 3x3x256
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2),             # pool5 1x1x256

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        self.embedding = nn.Sequential(
            nn.Linear(256, 4096),           # fc6
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate)
        )

        self.classifier = nn.Sequential(
            nn.Linear(4096*2, 4096),        # fc7
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),

            nn.Linear(4096, 4096),          # fc8
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),

            nn.Linear(4096, num_classes)    # fc9
        )

        self._initialize_weights()

    def _initialize_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
        
    def forward(self, x1, x2):
        feat1 = self.embedding(self.backbone(x1))
        feat2 = self.embedding(self.backbone(x2))

        fusion = torch.cat((feat1, feat2), dim=1)

        logits = self.classifier(fusion)
        
        return logits

In [11]:
test_vector = torch.zeros((32, 3, 96, 96))
model = LateFusionNetwork()
out = model(test_vector, test_vector)

RuntimeError: Calculated padded input size per channel: (1 x 1). Kernel size: (3 x 3). Kernel size can't be greater than actual input size

In [90]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.to(device)
    model.train()
    for batch, (X1, X2, y) in enumerate(dataloader):
        X1 = X1.to(device)
        X2 = X2.to(device)
        y = y.to(device)
        pred = model(X1, X2)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 40 == 0:
            loss, curr = loss.item(), batch*32 + len(X1)
            print(f"loss: {loss:>7f} [{curr:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X1, X2, y in dataloader:
            X1, X2, y = X1.to(device), X2.to(device), y.to(device)
            pred = model(X1, X2)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct)::>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [93]:
loss_fn = nn.CrossEntropyLoss()
model = LateFusionNetwork()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-----------------------------------")
    train_loop(training_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-----------------------------------
loss: 2.062957 [   32/ 5011]
loss: 2.106948 [ 1312/ 5011]
loss: 2.072778 [ 2592/ 5011]
loss: 2.082573 [ 3872/ 5011]
Test Error: 
 Accuracy: 12.8%, Avg loss: 2.086775 

Epoch 2
-----------------------------------
loss: 2.098771 [   32/ 5011]
loss: 2.125889 [ 1312/ 5011]
loss: 2.106571 [ 2592/ 5011]
loss: 2.068123 [ 3872/ 5011]
Test Error: 
 Accuracy: 12.0%, Avg loss: 2.081796 

Epoch 3
-----------------------------------
loss: 2.084861 [   32/ 5011]
loss: 2.074790 [ 1312/ 5011]
loss: 2.070066 [ 2592/ 5011]
loss: 2.088383 [ 3872/ 5011]
Test Error: 
 Accuracy: 13.1%, Avg loss: 2.079304 

Epoch 4
-----------------------------------
loss: 2.087549 [   32/ 5011]
loss: 2.099999 [ 1312/ 5011]
loss: 2.072521 [ 2592/ 5011]
loss: 2.080058 [ 3872/ 5011]
Test Error: 
 Accuracy: 12.6%, Avg loss: 2.080673 

Epoch 5
-----------------------------------
loss: 2.081096 [   32/ 5011]
loss: 2.086837 [ 1312/ 5011]
loss: 2.074700 [ 2592/ 5011]
loss: 2.070868 [ 3872

KeyboardInterrupt: 